# Dataset EDA — Mozilla *Lost in Transcription* (Indonesian–Javanese)

**Purpose.** Extract every fact about the three approved datasets that can be
learned from metadata and cheap audio probing, and emit it as one compact JSON
digest that can be handed back to Claude as project knowledge.

**How to run.**
1. Attach all three datasets: `jember-javanese-corpus`, `central-javanese-corpus`, `common-voice-corpus`.
2. Accelerator: **None**. This is a CPU job — do not burn GPU quota on it.
3. Internet: **On** (only cell 9 needs it, for the Whisper tokenizer).
4. Run All. Expect roughly 5–12 minutes, dominated by the Jember audio probe.
5. **Output tab → download all.** The archive already contains
   `findings/01-dataset-eda/`, so unzip it onto the repo root and every file
   lands in the right place — no manual sorting.

**Output convention.** Everything is written to `/kaggle/working/findings/<NOTEBOOK_ID>/`,
which mirrors `<repo>/findings/<NOTEBOOK_ID>/` exactly. Console output is teed to
`run-log.txt` automatically, so there is nothing to copy by hand. For a new notebook,
copy the OUTPUT CONVENTION block in cell 1 and change `NOTEBOOK_ID` only.

**Design note.** Every cell prints wide, readable tables *and* appends structured
values to a global `FINDINGS` dict. The prose output is for you; the JSON is for Claude.
Cells are independent enough that one failure does not sink the run — but they must
execute in order, since later cells reuse `J`, `CV_VOCAB`, `JEM_VOCAB`, `ID_LEX`.


## 0 — Dependencies

`mutagen` reads mp3 headers without decoding audio, which is what makes probing 200+ files cheap. Kaggle usually ships it; this is a no-op if so.

In [ ]:
!pip install -q mutagen 2>/dev/null || echo 'mutagen install skipped'

## 1 — Setup, path discovery, inventory

Resolves dataset roots by keyword so a slug rename does not break the notebook. Prints a bounded directory tree of each dataset — read this first to confirm nothing is missing.

In [ ]:
# ============================================================
# CELL 1 - SETUP, PATH DISCOVERY, INVENTORY
# ============================================================
import os, sys, re, json, glob, math, random, unicodedata, collections, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
random.seed(0); np.random.seed(0)

# Shared unicode-aware tokenizer: letters (incl. diacritics), internal - and '
RE_TOKEN = r"[^\W\d_]+(?:[-']+[^\W\d_]+)*"
# .webm matters: browser-recorded corpora ship Opus-in-WebM, not mp3.
AUDIO_EXT = (".mp3", ".wav", ".flac", ".m4a", ".ogg", ".opus",
             ".webm", ".aac", ".mp4", ".3gp", ".wma", ".amr")

# FINDINGS is the accumulator. Every cell writes into it.
# The last cell dumps it as one compact JSON block to paste back to Claude.
FINDINGS = {}
# NOTE: this reset is why partial re-runs do not work. Re-running THIS cell
# discards everything cells 3-9 collected. Always Run All.

def rec(section, **kw):
    FINDINGS.setdefault(section, {}).update(kw)

def require(val, name):
    # Fail with a sentence a human can act on, not a TypeError from pathlib.
    if val is None:
        raise RuntimeError(
            "%s is not resolved. Scroll up to the listing at the top of cell 1, "
            "then fill in the MANUAL OVERRIDE block and re-run cell 1." % name)
    return Path(val)

def head(title):
    print("\n" + "=" * 70); print(title); print("=" * 70)

def jsonable(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return round(float(o), 4)
    if isinstance(o, (np.bool_,)):    return bool(o)
    if isinstance(o, dict):  return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, Path): return str(o)
    return o

# ============================================================
# OUTPUT CONVENTION - mirrors the local repo layout
# ------------------------------------------------------------
# Kaggle writes to /kaggle/working. Everything this notebook produces goes to
#     /kaggle/working/findings/<NOTEBOOK_ID>/
# which is the same path as
#     <repo>/findings/<NOTEBOOK_ID>/
# so the downloaded Output archive can be dropped straight onto the repo root
# and every file lands where it belongs. Copy this block into every new
# notebook and change NOTEBOOK_ID only.
# ============================================================
NOTEBOOK_ID  = "01-dataset-eda"          # must match notebooks/<NOTEBOOK_ID>.ipynb
WORKING      = Path("/kaggle/working")
OUT_DIR      = WORKING / "findings" / NOTEBOOK_ID
ARTIFACT_DIR = OUT_DIR / "_kaggle-artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def out(filename):
    # Every write in this notebook goes through here. Nothing lands loose.
    return OUT_DIR / filename

# Tee stdout to a log file so the console output is saved without hand-copying.
# NOTE: in Jupyter the live stream is sys.stdout, not sys.__stdout__ - writing to
# the latter would send the output to the kernel process instead of the cell.
_ORIG_STDOUT = getattr(sys, "_eda_orig_stdout", sys.stdout)
sys._eda_orig_stdout = _ORIG_STDOUT

class _Tee:
    def __init__(self, path, stream):
        self.f = open(path, "w", encoding="utf-8"); self.s = stream
    def write(self, t):
        self.s.write(t); self.f.write(t); return len(t)
    def flush(self):
        try: self.s.flush()
        except Exception: pass
        self.f.flush()
    def isatty(self): return False

LOG_PATH = out("run-log.txt")
try:
    if isinstance(sys.stdout, _Tee): sys.stdout.f.close()
except Exception:
    pass
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)
print("output dir : %s" % OUT_DIR)
print("console log: %s  (restarts every time this cell runs)" % LOG_PATH)

INPUT = Path("/kaggle/input")

head("KAGGLE INPUT - WHAT IS ACTUALLY ATTACHED")
# Kaggle mounts inputs at different depths depending on how they were added:
#   /kaggle/input/<slug>/...                    (classic)
#   /kaggle/input/datasets/<user>/<slug>/...    (owner-scoped)
# Nothing below assumes a depth. Print the shallow layout for debugging.
def shallow(base, max_depth=4):
    out = []
    base = Path(base)
    for dp, dns, fns in os.walk(base):
        d = Path(dp)
        depth = len(d.relative_to(base).parts)
        if depth > max_depth:
            dns[:] = []
            continue
        if depth >= 1:
            out.append((depth, d, len(fns)))
        if len(fns) > 200:      # a clips/ dir - do not descend further
            dns[:] = []
    return out

if not INPUT.exists():
    print("!! /kaggle/input does not exist")
    layout = []
else:
    layout = shallow(INPUT)
    for depth, d, nf in layout[:60]:
        print("  %s%s%s" % ("    " * (depth - 1), d.name, ("  [%d files]" % nf) if nf else "/"))
    if len(layout) > 60:
        print("  ... +%d more directories" % (len(layout) - 60))

# ------------------------------------------------------------------
# Discovery is CONTENT-BASED and DEPTH-AGNOSTIC. A corpus is identified
# by a file only it has, then its root is derived from that file.
# ------------------------------------------------------------------
head("DISCOVERY")

def first_glob(*patterns):
    for pat in patterns:
        hits = sorted(INPUT.glob(pat))
        if hits:
            return hits[0]
    return None

# 1) Common Voice - the only corpus shipping clip_durations.tsv / validated.tsv
cv_hit = first_glob("**/clip_durations.tsv", "**/validated.tsv")
CV_DIR = cv_hit.parent if cv_hit else None
CV_DS  = CV_DIR.parent if CV_DIR else None
print("Common Voice   dir : %s" % (CV_DIR or "NOT FOUND"))

# 2) Jember - a TSV whose header carries start / end / text
jem_hits = []
for q in INPUT.glob("**/*.tsv"):
    if CV_DIR is not None and q.parent == CV_DIR:
        continue
    try:
        hdr = q.open(encoding="utf-8", errors="replace").readline().lower()
    except Exception:
        continue
    if "start" in hdr and "end" in hdr and "text" in hdr:
        jem_hits.append(q)
JEMBER_TSV = max(jem_hits, key=lambda q: q.stat().st_size) if jem_hits else None
JEMBER_DS  = JEMBER_TSV.parent if JEMBER_TSV else None
print("Jember         tsv : %s" % (JEMBER_TSV or "NOT FOUND"))

def _under(child, parent):
    if child is None or parent is None:
        return False
    try:
        Path(child).relative_to(Path(parent)); return True
    except ValueError:
        return False

# 3) Central Javanese TTS - the shallowest dir holding audio that belongs to
#    neither of the two corpora above.
skip = [d for d in (CV_DS, JEMBER_DS) if d is not None]
audio_dirs = []
for dp, dns, fns in os.walk(INPUT):
    d = Path(dp)
    if any(_under(d, s) for s in skip):
        dns[:] = []
        continue
    if any(Path(f).suffix.lower() in AUDIO_EXT for f in fns):
        audio_dirs.append(d)
        dns[:] = []               # found audio here, no need to go deeper
TTS_AUDIO_DIR = min(audio_dirs, key=lambda d: len(d.parts)) if audio_dirs else None
# Lift to the DATASET-level folder, otherwise sibling metadata one level up
# (a metadata.tsv next to the speaker folders) would be missed entirely.
# The other two corpora tell us where the dataset level is: it is the depth of
# their common container's children.
TTS_DS = TTS_AUDIO_DIR
if TTS_AUDIO_DIR is not None and CV_DS is not None and JEMBER_DS is not None:
    try:
        container = Path(os.path.commonpath([str(CV_DS), str(JEMBER_DS)]))
        rel = TTS_AUDIO_DIR.relative_to(container)
        if rel.parts:
            TTS_DS = container / rel.parts[0]
    except Exception:
        pass
elif TTS_AUDIO_DIR is not None:
    # no reference point - climb while the parent holds nothing but this branch
    while (TTS_AUDIO_DIR.parent != INPUT
           and len(TTS_DS.parent.relative_to(INPUT).parts) >= 1
           and sum(1 for _ in TTS_DS.parent.iterdir()) == 1):
        TTS_DS = TTS_DS.parent
print("Central JV TTS dir : %s" % (TTS_DS or "NOT FOUND"))
# Structural fallback. Audio detection fails whenever the corpus ships something
# unexpected - an unextracted archive, or an extension outside AUDIO_EXT. But the
# two corpora already resolved tell us where the dataset level sits, so the
# remaining sibling folder IS the third dataset, whatever happens to be inside it.
if TTS_DS is None and CV_DS is not None and JEMBER_DS is not None:
    try:
        container = Path(os.path.commonpath([str(CV_DS), str(JEMBER_DS)]))
        taken = {container / Path(x).relative_to(container).parts[0] for x in (CV_DS, JEMBER_DS)}
        cands = [c for c in sorted(container.iterdir()) if c.is_dir() and c not in taken]
        print("  no audio found; sibling folders left over:", [c.name for c in cands])
        if len(cands) == 1:
            TTS_DS = cands[0]
            print("  -> resolved structurally:", TTS_DS)
        elif len(cands) > 1:
            TTS_DS = cands[0]
            print("  -> AMBIGUOUS, guessed %s. Use MANUAL OVERRIDE if wrong." % TTS_DS.name)
    except Exception as e:
        print("  structural fallback failed:", e)

# ==================================================================
# MANUAL OVERRIDE - only if a line above still says NOT FOUND.
# Copy the real path from the layout listing at the top of this cell.
# ==================================================================
# JEMBER_DS  = Path("/kaggle/input/.../Jember Javanese Spontaneous Speech Corpus")
# JEMBER_TSV = JEMBER_DS / "Jember Javanese Spontaneous Speech Corpus - 1-200.tsv"
# CV_DIR     = Path("/kaggle/input/.../common-voice-corpus/id")
# CV_DS      = CV_DIR.parent
# TTS_DS     = Path("/kaggle/input/.../central-javanese-corpus")

print("")
print("RESOLVED:")
print("  JEMBER_DS  =", JEMBER_DS)
print("  JEMBER_TSV =", JEMBER_TSV)
print("  CV_DIR     =", CV_DIR)
print("  TTS_DS     =", TTS_DS)
missing = [n for n, v in [("JEMBER_DS", JEMBER_DS), ("CV_DIR", CV_DIR), ("TTS_DS", TTS_DS)] if v is None]
if missing:
    print("")
    print("!!! UNRESOLVED: %s" % missing)
    print("!!! Fill them in the MANUAL OVERRIDE block above and re-run this cell.")
else:
    print("")
    print("all three datasets resolved - safe to run the rest.")

def tree(root, max_depth=3, max_per_dir=8):
    # Print a bounded directory tree. Large dirs are summarised, not listed.
    root = Path(root)
    def walk(d, depth, prefix):
        if depth > max_depth: return
        try:
            entries = sorted(d.iterdir(), key=lambda x: (x.is_file(), x.name))
        except Exception as e:
            print(prefix + "  <unreadable: %s>" % e); return
        dirs  = [e for e in entries if e.is_dir()]
        files = [e for e in entries if e.is_file()]
        for e in dirs:
            try:    n = sum(1 for _ in e.iterdir())
            except Exception: n = -1
            print("%s%s/  (%d entries)" % (prefix, e.name, n))
            walk(e, depth + 1, prefix + "    ")
        for e in files[:max_per_dir]:
            print("%s%s  [%.1f KB]" % (prefix, e.name, e.stat().st_size / 1024))
        if len(files) > max_per_dir:
            exts = collections.Counter(f.suffix.lower() for f in files)
            print("%s... +%d more files  %s" % (prefix, len(files) - max_per_dir, dict(exts)))
    print(root)
    walk(root, 1, "  ")

for name, ds in [("JEMBER", JEMBER_DS), ("CENTRAL-JAVANESE-TTS", TTS_DS), ("COMMON-VOICE", CV_DS)]:
    head("TREE: %s" % name)
    if ds is None: print("(skipped - not resolved)")
    else:          tree(ds)

rec("env",
    kaggle_layout=[str(d.relative_to(INPUT)) for _, d, _ in layout[:40]],
    jember_root=str(JEMBER_DS), cv_root=str(CV_DS), cv_dir=str(CV_DIR), tts_root=str(TTS_DS),
    unresolved=missing, python=sys.version.split()[0], pandas=pd.__version__)

## 2 — Audio probe helper

Duration, sample rate and channel count via header reads, with a mutagen → soundfile → librosa fallback chain. Never raises; failures are counted and reported.

In [ ]:
# ============================================================
# CELL 2 - AUDIO PROBE HELPER
# Duration / samplerate / channels for mp3 & wav, with fallbacks.
# ============================================================
_probe_backend = None
try:
    from mutagen import File as _MutagenFile
    _probe_backend = "mutagen"
except Exception:
    _MutagenFile = None
try:
    import soundfile as _sf
except Exception:
    _sf = None
try:
    import librosa as _librosa
except Exception:
    _librosa = None

print("probe backends -> mutagen:", _MutagenFile is not None,
      "| soundfile:", _sf is not None, "| librosa:", _librosa is not None)

import subprocess, shutil as _shutil
_FFPROBE = _shutil.which("ffprobe")
print("ffprobe:", _FFPROBE or "NOT FOUND (webm/opus durations will be unavailable)")

def _probe_ffprobe(p):
    # The only backend here that reads WebM/Matroska. Header-only, so fast.
    if not _FFPROBE:
        return None
    try:
        r = subprocess.run([_FFPROBE, "-v", "quiet", "-print_format", "json",
                            "-show_format", "-show_streams", str(p)],
                           capture_output=True, timeout=20)
        j = json.loads(r.stdout.decode("utf-8", "replace"))
        au = next((st for st in j.get("streams", []) if st.get("codec_type") == "audio"), None)
        dur = j.get("format", {}).get("duration") or (au or {}).get("duration")
        if dur is None:
            return None
        return dict(dur_s=float(dur),
                    sr=int(au["sample_rate"]) if au and au.get("sample_rate") else None,
                    ch=au.get("channels") if au else None,
                    bitrate=int(j["format"]["bit_rate"]) if j.get("format", {}).get("bit_rate") else None,
                    codec=(au or {}).get("codec_name"), backend="ffprobe")
    except Exception:
        return None

def probe(path):
    # Returns dict(dur_s, sr, ch, backend) or None. Never raises.
    p = str(path)
    if _MutagenFile is not None:
        try:
            m = _MutagenFile(p)
            if m is not None and getattr(m, "info", None) is not None:
                i = m.info
                d = getattr(i, "length", None)
                if d:
                    return dict(dur_s=float(d), sr=getattr(i, "sample_rate", None),
                                ch=getattr(i, "channels", None),
                                bitrate=getattr(i, "bitrate", None), backend="mutagen")
        except Exception:
            pass
    if _sf is not None:
        try:
            info = _sf.info(p)
            return dict(dur_s=float(info.duration), sr=info.samplerate,
                        ch=info.channels, bitrate=None, backend="soundfile")
        except Exception:
            pass
    r = _probe_ffprobe(p)
    if r is not None:
        return r
    if _librosa is not None:
        try:
            d = _librosa.get_duration(path=p)
            return dict(dur_s=float(d), sr=None, ch=None, bitrate=None, backend="librosa")
        except Exception:
            pass
    return None

def probe_many(paths, label=""):
    # Probe a list of files, report failures. Bounded work: caller decides size.
    out, fail = [], 0
    for i, p in enumerate(paths):
        r = probe(p)
        if r is None:
            fail += 1
        else:
            r["path"] = str(p); out.append(r)
    if label:
        print("%s: probed %d ok, %d failed" % (label, len(out), fail))
    return pd.DataFrame(out)

def describe_dur(series, name):
    # Compact distribution summary used everywhere below.
    s = pd.Series(series).dropna().astype(float)
    if len(s) == 0:
        print("%s: EMPTY" % name); return {}
    q = s.quantile([.01, .25, .5, .75, .95, .99]).round(2).to_dict()
    d = dict(n=int(len(s)), total_h=round(s.sum() / 3600, 3), mean_s=round(s.mean(), 2),
             min_s=round(s.min(), 2), p01=q[.01], p25=q[.25], median=q[.5],
             p75=q[.75], p95=q[.95], p99=q[.99], max_s=round(s.max(), 2),
             n_over_30s=int((s > 30).sum()), n_under_1s=int((s < 1).sum()),
             n_nonpositive=int((s <= 0).sum()))
    print(name, "->", json.dumps(d))
    return d

## 3 — Jember: segments and timing integrity

The main corpus. Checks segment durations, unparseable timestamps, non-positive durations, and overlapping segments within a file — all of which silently corrupt training data if left in.

In [ ]:
# ============================================================
# CELL 3 - JEMBER: segments, timing integrity, coverage vs audio
# ============================================================
head("JEMBER - FILE INVENTORY")
_jroot = require(JEMBER_DS, "JEMBER_DS")
jem_files = [p for p in _jroot.rglob("*") if p.is_file()]
jem_tsv   = [p for p in jem_files if p.suffix.lower() in (".tsv", ".csv")]
jem_audio = [p for p in jem_files if p.suffix.lower() in (".mp3", ".wav", ".flac", ".m4a", ".ogg")]
jem_other = [p for p in jem_files if p not in jem_tsv and p not in jem_audio]
print("tabular:", [p.name for p in jem_tsv])
print("audio  :", len(jem_audio), "files, exts:", collections.Counter(p.suffix.lower() for p in jem_audio))
print("other  :", [p.name for p in jem_other][:20], "(+%d)" % max(0, len(jem_other) - 20))
print("\n>>> ANY file above that is not the TSV or mp3 may carry SPEAKER METADATA - check it.")

# --- load the annotation table -------------------------------------------
jt = JEMBER_TSV if JEMBER_TSV is not None else max(jem_tsv, key=lambda p: p.stat().st_size)
print("\nloading:", jt.name)
J = pd.read_csv(jt, sep="\t", dtype=str, keep_default_na=False)
print("shape:", J.shape, "| columns:", list(J.columns))
print("\n--- first 5 rows ---"); print(J.head(5).to_string())
print("\n--- last 3 rows ---");  print(J.tail(3).to_string())

col_file = J.columns[0]; col_start, col_end, col_text = "start", "end", "text"

def hms(t):
    # "H:MM:SS" or "H:MM:SS.mmm" -> seconds. Returns nan on junk.
    try:
        parts = str(t).strip().split(":")
        if len(parts) == 3:   h, m, s = parts
        elif len(parts) == 2: h, (m, s) = 0, parts
        else: return np.nan
        return int(h) * 3600 + int(m) * 60 + float(s)
    except Exception:
        return np.nan

J["start_s"] = J[col_start].map(hms)
J["end_s"]   = J[col_end].map(hms)
J["dur_s"]   = J["end_s"] - J["start_s"]
J["fid"]     = J[col_file].astype(str).str.strip()

head("JEMBER - SEGMENT DURATION DISTRIBUTION")
jem_dur = describe_dur(J["dur_s"], "jember_segments")

head("JEMBER - TIMING INTEGRITY")
bad_parse = int(J["start_s"].isna().sum() + J["end_s"].isna().sum())
nonpos    = J[J["dur_s"] <= 0]
print("unparseable timestamps :", bad_parse)
print("segments with dur <= 0 :", len(nonpos))
if len(nonpos):
    print("\n--- ALL non-positive-duration rows (these must be dropped or fixed) ---")
    print(nonpos[[col_file, col_start, col_end, "dur_s", col_text]].head(50).to_string())

# Overlap / ordering check within each audio file.
ov_rows, gap_stats = [], []
for fid, g in J.sort_values([ "fid", "start_s"]).groupby("fid", sort=False):
    g = g.dropna(subset=["start_s", "end_s"])
    if len(g) < 2: continue
    prev_end = g["end_s"].shift(1)
    gaps = g["start_s"] - prev_end
    ov = gaps[gaps < -0.001]
    if len(ov): ov_rows.append((fid, len(ov), round(float(ov.min()), 2)))
    gap_stats.extend(gaps.dropna().tolist())
print("\nfiles containing overlapping segments:", len(ov_rows))
for r in ov_rows[:15]: print("   file=%s n_overlaps=%d worst=%.2fs" % r)
gs = pd.Series(gap_stats)
print("\ninter-segment gap (s): median=%.2f p95=%.2f max=%.2f | negative(overlap)=%d"
      % (gs.median(), gs.quantile(.95), gs.max(), int((gs < 0).sum())))

head("JEMBER - SEGMENTS PER AUDIO FILE")
per = J.groupby("fid").agg(n_seg=("dur_s", "size"), sum_s=("dur_s", "sum"),
                           first_s=("start_s", "min"), last_e=("end_s", "max"))
print(per.describe().round(2).to_string())
print("\ntop 5 longest files by annotated time:")
print(per.sort_values("sum_s", ascending=False).head(5).round(2).to_string())

rec("jember",
    n_segments=len(J), n_audio_files=int(J["fid"].nunique()),
    n_mp3_on_disk=len(jem_audio), columns=list(J.columns[:4]),
    duration=jem_dur, bad_timestamp_parse=bad_parse, n_nonpositive_dur=len(nonpos),
    n_files_with_overlap=len(ov_rows),
    gap_median_s=round(float(gs.median()), 2), gap_p95_s=round(float(gs.quantile(.95)), 2),
    seg_per_file_median=float(per["n_seg"].median()),
    extra_files=[p.name for p in jem_other][:20])

## 4 — Jember: audio files vs annotation coverage

Probes all mp3 files and compares real audio length against annotated span. The gap is unlabelled in-domain audio, which is the cheapest source of extra signal via pseudo-labelling.

In [ ]:
# ============================================================
# CELL 4 - JEMBER: audio files vs annotation coverage
# 208 files is small enough to probe exhaustively (~1-2 min).
# ============================================================
head("JEMBER - AUDIO PROBE (all files)")
jem_audio_map = {}
for p in jem_audio:
    jem_audio_map[p.stem.strip()] = p          # "1.mp3" -> key "1"
print("audio files on disk:", len(jem_audio_map))

A = probe_many(sorted(jem_audio, key=lambda p: p.name), label="jember_audio")
if len(A):
    A["fid"] = A["path"].map(lambda s: Path(s).stem.strip())
    print("\nsample rates :", dict(collections.Counter(A["sr"].dropna().astype(int))))
    print("channels     :", dict(collections.Counter(A["ch"].dropna().astype(int))))
    if A["bitrate"].notna().any():
        print("bitrate kbps :", dict(collections.Counter((A["bitrate"].dropna() // 1000).astype(int))))
    jem_file_dur = describe_dur(A["dur_s"], "jember_audio_files")

    head("JEMBER - ANNOTATION COVERAGE")
    cov = per.join(A.set_index("fid")["dur_s"].rename("audio_s"), how="outer")
    cov["coverage"] = cov["sum_s"] / cov["audio_s"]
    cov["unannotated_s"] = cov["audio_s"] - cov["sum_s"]
    total_audio = float(cov["audio_s"].sum(skipna=True))
    total_annot = float(cov["sum_s"].sum(skipna=True))
    print("total audio on disk      : %.2f h" % (total_audio / 3600))
    print("total annotated segments : %.2f h" % (total_annot / 3600))
    print("overall coverage         : %.1f %%" % (100 * total_annot / max(total_audio, 1e-9)))
    print("UNANNOTATED audio        : %.2f h  <-- candidate for self-training / pseudo-labelling"
          % ((total_audio - total_annot) / 3600))
    print("\ncoverage per file:"); print(cov["coverage"].describe().round(3).to_string())
    miss_tsv   = cov[cov["n_seg"].isna()]
    miss_audio = cov[cov["audio_s"].isna()]
    print("\naudio files with NO rows in the TSV :", len(miss_tsv), list(miss_tsv.index)[:20])
    print("TSV file-ids with NO audio on disk  :", len(miss_audio), list(miss_audio.index)[:20])
    worst = cov.dropna(subset=["coverage"]).sort_values("coverage").head(8)
    print("\nlowest-coverage files (possible truncated annotation):")
    print(worst[["n_seg", "sum_s", "audio_s", "coverage"]].round(2).to_string())

    rec("jember_audio",
        n_probed=len(A), sample_rates=jsonable(dict(collections.Counter(A["sr"].dropna().astype(int)))),
        channels=jsonable(dict(collections.Counter(A["ch"].dropna().astype(int)))),
        file_duration=jem_file_dur,
        total_audio_h=round(total_audio / 3600, 2), total_annotated_h=round(total_annot / 3600, 2),
        coverage_pct=round(100 * total_annot / max(total_audio, 1e-9), 1),
        unannotated_h=round((total_audio - total_annot) / 3600, 2),
        n_audio_without_tsv=len(miss_tsv), n_tsv_without_audio=len(miss_audio))
else:
    print("!! audio probing failed entirely - install mutagen:  !pip install mutagen -q")
    rec("jember_audio", error="probe_failed")

## 5 — Jember: orthography, tokens, disfluency

Character inventory, token statistics, annotation conventions (brackets, truncation markers, reduplication) and filler words. This determines the text-normalisation pipeline.

In [ ]:
# ============================================================
# CELL 5 - JEMBER: orthography, tokens, disfluency, reduplication
# This drives text normalisation and tokenizer decisions.
# ============================================================
head("JEMBER - RAW TEXT SAMPLE (20 consecutive segments)")
for _, r in J.iloc[100:120].iterrows():
    print("[%s %s-%s] %s" % (r["fid"], r["start"], r["end"], r["text"]))

txt = J[col_text].astype(str)
head("JEMBER - CHARACTER INVENTORY")
chars = collections.Counter("".join(txt.tolist()))
print("distinct characters:", len(chars))
rows = []
for ch, n in sorted(chars.items(), key=lambda kv: -kv[1]):
    cat = unicodedata.category(ch)
    nm  = unicodedata.name(ch, "?")
    rows.append(dict(char=repr(ch), n=n, cat=cat, name=nm[:38]))
CH = pd.DataFrame(rows)
print("\n--- all non-ASCII-letter / non-space characters (the ones that matter) ---")
special = CH[~CH["char"].str.strip("'").str.match(r"^[A-Za-z ]$", na=False)]
print(special.to_string(index=False))
nonascii = sorted([c for c in chars if ord(c) > 127])
print("\nNON-ASCII chars present:", [(c, chars[c], unicodedata.name(c, "?")) for c in nonascii])

head("JEMBER - TOKEN STATISTICS")
def toks(s): return re.findall(RE_TOKEN, s)
J["tok"]   = txt.map(toks)
J["ntok"]  = J["tok"].map(len)
J["nchar"] = txt.map(len)
print("tokens/segment:", J["ntok"].describe().round(2).to_dict())
print("chars/segment :", J["nchar"].describe().round(2).to_dict())
J["cps"] = J["nchar"] / J["dur_s"].replace(0, np.nan)
print("chars/sec     : median=%.2f p95=%.2f  (outliers = misaligned timestamps)"
      % (J["cps"].median(), J["cps"].quantile(.95)))
print("empty-text segments:", int((txt.str.strip() == "").sum()))
all_tokens = [t.lower() for lst in J["tok"] for t in lst]
vocab = collections.Counter(all_tokens)
print("total tokens: %d | vocab size: %d | hapax: %d (%.1f%%)"
      % (len(all_tokens), len(vocab), sum(1 for v in vocab.values() if v == 1),
         100 * sum(1 for v in vocab.values() if v == 1) / max(len(vocab), 1)))
print("\ntop 40 tokens:", vocab.most_common(40))

head("JEMBER - DISFLUENCY / ANNOTATION CONVENTIONS")
pats = {
    "square_bracket": r"\[[^\]]*\]", "paren": r"\([^)]*\)", "angle": r"<[^>]*>",
    "curly": r"\{[^}]*\}", "trailing_hyphen_truncation": r"\b\w+-(?=\s|$)",
    "ellipsis": r"\.\.\.|…", "digit": r"\d", "uppercase_run": r"\b[A-Z]{2,}\b",
    "multi_space": r"  +", "reduplication": r"\b(\w+)-\1\b",
    "redup_partial": r"\b\w+-\w+\b",
}
for name, pat in pats.items():
    hits = txt.str.findall(pat)
    n_seg = int((hits.map(len) > 0).sum())
    flat  = [h if isinstance(h, str) else h for lst in hits for h in lst]
    ex    = collections.Counter([str(x) for x in flat]).most_common(8)
    print("%-28s segments=%5d  examples=%s" % (name, n_seg, ex))

fillers = ["e", "ee", "eee", "em", "emm", "mm", "hmm", "anu", "apa", "opo", "iku", "ini",
           "itu", "eh", "ah", "oh", "lah", "ya", "yo", "nah", "kan", "gitu", "ngono"]
fc = {f: vocab.get(f, 0) for f in fillers if vocab.get(f, 0) > 0}
print("\nfiller-ish token counts:", dict(sorted(fc.items(), key=lambda kv: -kv[1])))

rec("jember_text",
    n_distinct_chars=len(chars),
    non_ascii_chars=[[c, chars[c], unicodedata.name(c, "?")] for c in nonascii],
    special_chars=jsonable({r["char"]: r["n"] for _, r in special.head(40).iterrows()}),
    tokens_per_seg_median=float(J["ntok"].median()),
    chars_per_sec_median=round(float(J["cps"].median()), 2),
    total_tokens=len(all_tokens), vocab_size=len(vocab),
    hapax_pct=round(100 * sum(1 for v in vocab.values() if v == 1) / max(len(vocab), 1), 1),
    top40=vocab.most_common(40),
    n_empty_text=int((txt.str.strip() == "").sum()),
    fillers=fc)
JEM_VOCAB = vocab

head("JEMBER - DIACRITIC CONSISTENCY (pepet / taling)")
# Javanese Latin orthography marks the pepet and the open-e. If the corpus uses
# more than one codepoint for the same sound, the model is being asked to learn
# a distinction the annotators did not apply consistently - and every mismatch
# is a WER error that has nothing to do with acoustics.
EVAR = {"e": "e", "\u00e8": "e-grave", "\u00e9": "e-acute", "\u00ea": "e-circumflex", "\u011b": "e-caron"}
present = {c: chars.get(c, 0) for c in EVAR if chars.get(c, 0)}
print("e-family codepoints in use:", {("%s (%s)" % (c, EVAR[c])): n for c, n in present.items()})

def fold(w):
    return (w.replace("\u00e8", "e").replace("\u00e9", "e")
             .replace("\u00ea", "e").replace("\u011b", "e"))

groups = collections.defaultdict(collections.Counter)
for w, n in JEM_VOCAB.items():
    groups[fold(w)][w] += n
amb = {k: v for k, v in groups.items() if len(v) > 1}
print("")
print("folded forms written more than one way: %d" % len(amb))
tok_at_risk = sum(sum(v.values()) for v in amb.values())
print("token mass affected: %d (%.1f %% of all tokens)"
      % (tok_at_risk, 100 * tok_at_risk / max(sum(JEM_VOCAB.values()), 1)))
print("")
print("worst 30 inconsistently-spelled words:")
for k, v in sorted(amb.items(), key=lambda kv: -sum(kv[1].values()))[:30]:
    print("   %-18s %s" % (k, dict(v.most_common())))
print("")
print("vocab size raw: %d -> after folding e-diacritics: %d (-%.1f %%)"
      % (len(JEM_VOCAB), len(groups), 100 * (1 - len(groups) / max(len(JEM_VOCAB), 1))))
print(">>> DECISION NEEDED: does the competition metric compare raw or normalised text?")
print(">>> If raw, these spellings must be predicted exactly and folding would hurt.")
print(">>> If normalised, fold them in training to stop wasting model capacity.")

rec("jember_orthography",
    e_family_counts=jsonable({EVAR[c]: n for c, n in present.items()}),
    n_ambiguous_forms=len(amb),
    ambiguous_token_mass_pct=round(100 * tok_at_risk / max(sum(JEM_VOCAB.values()), 1), 1),
    vocab_raw=len(JEM_VOCAB), vocab_folded=len(groups),
    worst_examples=jsonable({k: dict(v.most_common())
                             for k, v in sorted(amb.items(), key=lambda kv: -sum(kv[1].values()))[:20]}))

## 6 — Common Voice Indonesian 26.0

Split sizes in hours, speaker counts, speaker overlap between official splits, speaker concentration, demographics and vote quality. The speaker column is the one to watch.

In [ ]:
# ============================================================
# CELL 6 - COMMON VOICE id 26.0: splits, speakers, hours, demographics
# ============================================================
CVD = Path(CV_DIR) if CV_DIR is not None else None
if CVD is None:
    _r = require(CV_DS, "CV_DS")
    CVD = next((c.parent for c in _r.rglob("validated.tsv")), None)
require(CVD, "CV_DIR")
print("CV data dir:", CVD)

head("COMMON VOICE - README")
rm = CVD / "README.md"
if rm.exists():
    print(rm.read_text(encoding="utf-8", errors="replace")[:2500])

dur = pd.read_csv(CVD / "clip_durations.tsv", sep="\t")
dur.columns = ["path", "ms"][:len(dur.columns)]
DMAP = dict(zip(dur["path"], dur["ms"]))
print("\nclip_durations rows:", len(dur))

SPLITS = ["validated", "train", "dev", "test", "other", "invalidated", "reported"]
CV = {}
for s in SPLITS:
    f = CVD / ("%s.tsv" % s)
    if f.exists():
        CV[s] = pd.read_csv(f, sep="\t", low_memory=False)
print("\nloaded splits:", {k: v.shape for k, v in CV.items()})
print("\ncolumns of validated:", list(CV["validated"].columns))

head("COMMON VOICE - SPLIT SUMMARY")
rows = []
for s, df in CV.items():
    if "path" not in df.columns: continue
    hrs = sum(DMAP.get(p, 0) for p in df["path"]) / 3.6e6
    rows.append(dict(split=s, clips=len(df), hours=round(hrs, 2),
                     speakers=df["client_id"].nunique() if "client_id" in df else None,
                     clips_per_spk=round(len(df) / max(df["client_id"].nunique(), 1), 1)
                     if "client_id" in df else None,
                     sentences=df["sentence"].nunique() if "sentence" in df else None))
S = pd.DataFrame(rows)
print(S.to_string(index=False))
print("\n>>> WATCH THE speakers COLUMN. A train split with a handful of speakers is")
print(">>> nearly worthless for acoustic diversity - build your own split from validated.")

head("COMMON VOICE - SPEAKER OVERLAP BETWEEN OFFICIAL SPLITS")
sets = {s: set(df["client_id"]) for s, df in CV.items() if "client_id" in df}
keys = [k for k in ["train", "dev", "test", "other", "invalidated"] if k in sets]
O = pd.DataFrame(index=keys, columns=keys, dtype=int)
for a in keys:
    for b in keys:
        O.loc[a, b] = len(sets[a] & sets[b])
print(O.to_string())

head("COMMON VOICE - SPEAKER CONCENTRATION (validated)")
V = CV["validated"].copy()
V["ms"] = V["path"].map(DMAP).fillna(0)
spk = V.groupby("client_id")["ms"].agg(["size", "sum"]).sort_values("sum", ascending=False)
spk["hours"] = spk["sum"] / 3.6e6
tot = spk["hours"].sum()
for k in [1, 5, 10, 25, 50, 100]:
    if k <= len(spk):
        print("top %3d speakers hold %5.1f %% of validated hours" % (k, 100 * spk["hours"].head(k).sum() / tot))
print("\nspeakers with >=10 clips: %d | >=50 clips: %d | >=200 clips: %d"
      % ((spk["size"] >= 10).sum(), (spk["size"] >= 50).sum(), (spk["size"] >= 200).sum()))

head("COMMON VOICE - CLIP DURATION + DEMOGRAPHICS (validated)")
cv_dur = describe_dur(V["ms"] / 1000.0, "cv_validated_clips")
for c in ["age", "gender", "accents", "variant", "locale", "sentence_domain", "segment"]:
    if c in V.columns:
        vc = V[c].fillna("<NA>").value_counts().head(10)
        print("\n%s:" % c); print(vc.to_string())

head("COMMON VOICE - VOTE QUALITY")
if "up_votes" in V.columns:
    print("up_votes  :", V["up_votes"].describe().round(2).to_dict())
    print("down_votes:", V["down_votes"].describe().round(2).to_dict())
    print("clips with >=1 down_vote: %d (%.1f%%)"
          % ((V["down_votes"] >= 1).sum(), 100 * (V["down_votes"] >= 1).mean()))

head("COMMON VOICE - SENTENCE TEXT")
sent = V["sentence"].astype(str)
print("distinct sentences: %d / %d clips  (repeat factor %.2f)"
      % (sent.nunique(), len(sent), len(sent) / max(sent.nunique(), 1)))
cvtok = [t.lower() for s_ in sent.drop_duplicates() for t in re.findall(RE_TOKEN, s_)]
CV_VOCAB = collections.Counter(cvtok)
print("unique-sentence tokens: %d | vocab: %d" % (len(cvtok), len(CV_VOCAB)))
print("top 30:", CV_VOCAB.most_common(30))
cvchars = collections.Counter("".join(sent.drop_duplicates().tolist()))
print("\nnon-ASCII chars in CV sentences:",
      [(c, cvchars[c]) for c in sorted(cvchars) if ord(c) > 127][:30])
print("\nsample sentences:")
for s_ in sent.drop_duplicates().head(8): print("  -", s_)

rec("common_voice",
    data_dir=str(CVD), splits=jsonable(S.to_dict("records")),
    speaker_overlap=jsonable(O.to_dict()),
    validated_clip_duration=cv_dur,
    top10_speaker_share_pct=round(100 * spk["hours"].head(10).sum() / tot, 1),
    n_speakers_ge50_clips=int((spk["size"] >= 50).sum()),
    distinct_sentences=int(sent.nunique()), sentence_repeat_factor=round(len(sent) / max(sent.nunique(), 1), 2),
    cv_vocab_size=len(CV_VOCAB),
    demographics={c: jsonable(V[c].fillna("<NA>").value_counts().head(6).to_dict())
                  for c in ["age", "gender", "accents", "variant"] if c in V.columns})

## 7 — Central Javanese TTS

Structure is unknown ahead of time, so this cell discovers it: extensions, metadata files, directory layout, filename patterns, and any transcript column it can find.

In [ ]:
# ============================================================
# CELL 7 - CENTRAL JAVANESE TTS: structure discovery
# Contents unknown locally (it was a nested zip), so discover generically.
# ============================================================
import shutil, tempfile

def _rel(q, base):
    # TTS_DS moves when an archive is unpacked, so paths collected before the
    # move are no longer relative to it. Degrade to the full path, never raise.
    try:
        return str(Path(q).relative_to(base))
    except Exception:
        return str(q)

head("CENTRAL JAVANESE TTS - FULL INVENTORY")
TTS_VOCAB = collections.Counter()
tts_txt = []
if TTS_DS is None:
    print("!! TTS_DS not resolved - set it in the MANUAL OVERRIDE block in cell 1.")
    rec("tts_central", error="not_found")
else:
    tf = [p for p in Path(TTS_DS).rglob("*") if p.is_file()]
    print("total files:", len(tf))
    print("by extension:", dict(collections.Counter(p.suffix.lower() for p in tf)))
    t_audio = [p for p in tf if p.suffix.lower() in AUDIO_EXT]
    t_arch  = [p for p in tf if p.suffix.lower() in (".zip", ".gz", ".tar", ".7z", ".rar", ".bz2", ".xz")]
    META_EXT = (".tsv", ".csv", ".txt", ".json", ".jsonl", ".xlsx", ".md")
    t_meta  = [p for p in tf if p.suffix.lower() in META_EXT]
    # Kaggle sometimes leaves a dataset packed. Unpack it here rather than
    # forcing another notebook round-trip. Extraction goes to scratch space,
    # NOT to /kaggle/working, so the Output download stays small.
    if t_arch and not t_audio:
        print("")
        print("!! UNEXTRACTED ARCHIVES - unpacking to scratch space:")
        for q in t_arch[:10]:
            print("   %s  [%.1f MB]" % (_rel(q, TTS_DS), q.stat().st_size / 1048576))
        scratch = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path(tempfile.mkdtemp())
        unpack = scratch / "tts_unpacked"
        unpack.mkdir(parents=True, exist_ok=True)
        for q in t_arch:
            try:
                shutil.unpack_archive(str(q), str(unpack))
                print("   unpacked:", q.name)
            except Exception as e:
                print("   FAILED %s: %s" % (q.name, e))
        tf = [x for x in unpack.rglob("*") if x.is_file()]
        t_audio = [x for x in tf if x.suffix.lower() in AUDIO_EXT]
        t_meta  = [x for x in tf if x.suffix.lower() in META_EXT]
        TTS_DS  = unpack
        print("   after unpacking: %d files, %d audio" % (len(tf), len(t_audio)))
        print("   by extension:", dict(collections.Counter(x.suffix.lower() for x in tf)))
    elif t_arch:
        print("")
        print("!! archives present alongside audio (left packed):",
              [q.name for q in t_arch[:5]])

    if not t_audio:
        print("")
        print("!! no audio matched %s" % (AUDIO_EXT,))
        print("   full extension census:", dict(collections.Counter(x.suffix.lower() for x in tf)))
        print("   deepest paths seen:")
        for q in sorted(tf, key=lambda z: -len(z.parts))[:15]:
            print("    ", _rel(q, TTS_DS))

    print("")
    print("metadata-ish files:")
    for q in t_meta[:40]:
        print("   %s  [%.1f KB]" % (_rel(q, TTS_DS), q.stat().st_size / 1024))
    if not t_meta:
        print("   (none)")

    # Dump a couple of metadata files in full - the schema is unknown up front
    # and this is the only way to see what the columns actually mean.
    for q in t_meta[:3]:
        head("TTS METADATA: %s" % _rel(q, TTS_DS))
        try:
            if q.suffix.lower() in (".tsv", ".csv"):
                dd = pd.read_csv(q, sep="	" if q.suffix.lower() == ".tsv" else ",",
                                 dtype=str, keep_default_na=False, on_bad_lines="skip")
                print("shape:", dd.shape, "| columns:", list(dd.columns))
                print(dd.head(8).to_string())
            else:
                print(q.read_text(encoding="utf-8", errors="replace")[:1200])
        except Exception as e:
            print("  read error:", e)

    head("TTS - AUDIO PROBE (sample of 200)")
    print("audio files:", len(t_audio))
    tts_dur = {}
    if t_audio:
        samp = random.sample(t_audio, min(200, len(t_audio)))
        TA = probe_many(samp, label="tts_audio_sample")
        if len(TA):
            print("sample rates:", dict(collections.Counter(TA["sr"].dropna().astype(int))))
            print("channels    :", dict(collections.Counter(TA["ch"].dropna().astype(int))))
            tts_dur = describe_dur(TA["dur_s"], "tts_clip_sample")
            est_h = TA["dur_s"].mean() * len(t_audio) / 3600
            print("estimated total: %.2f h (mean clip %.2fs x %d files)"
                  % (est_h, TA["dur_s"].mean(), len(t_audio)))
            tts_dur["estimated_total_h"] = round(float(est_h), 2)

    head("TTS - DIRECTORY / SPEAKER STRUCTURE")
    parents = collections.Counter(_rel(p.parent, TTS_DS) for p in t_audio)
    print("distinct audio-holding directories:", len(parents))
    for k, v in list(parents.most_common(20)):
        print("   %-60s %d files" % (k[:60], v))
    stems = [p.stem for p in t_audio[:2000]]
    print("\nfilename stem examples:", stems[:12])
    pref = collections.Counter(re.split(r"[_0-9-]", s)[0] for s in stems if s)
    print("stem prefixes (possible speaker ids):", pref.most_common(15))

    # Pick the transcript column properly. The naive "longest column" heuristic
    # picks the filename column too - 32-char hashes are long but are not text -
    # which poisons the vocabulary with hex digits. Require real word structure.
    def looks_like_text(series):
        v = series.astype(str).head(200)
        if len(v) == 0:
            return False
        if v.str.lower().str.endswith(AUDIO_EXT).mean() > 0.5:
            return False                       # filename column
        if v.str.fullmatch(r"[0-9a-fA-F]{8,}").mean() > 0.5:
            return False                       # bare hash column
        has_space = v.str.contains(r"\s").mean()
        mean_len  = v.str.len().mean()
        return has_space > 0.5 and mean_len > 12

    tts_pairs = []          # (audio filename, transcript)
    for q in t_meta:
        if q.suffix.lower() not in (".tsv", ".csv"):
            continue
        try:
            dd = pd.read_csv(q, sep="	" if q.suffix.lower() == ".tsv" else ",",
                             dtype=str, keep_default_na=False, on_bad_lines="skip")
        except Exception:
            continue
        text_cols = [c for c in dd.columns if looks_like_text(dd[c])]
        file_cols = [c for c in dd.columns
                     if dd[c].astype(str).str.lower().str.endswith(AUDIO_EXT).mean() > 0.5]
        if not text_cols:
            continue
        tc = text_cols[0]
        fc = file_cols[0] if file_cols else None
        tts_txt.extend(dd[tc].astype(str).tolist())
        if fc:
            tts_pairs.extend(zip(dd[fc].astype(str), dd[tc].astype(str)))

    if tts_txt:
        head("TTS - TRANSCRIPT TEXT")
        print("transcript rows: %d | rows paired to an audio filename: %d"
              % (len(tts_txt), len(tts_pairs)))
        if tts_pairs:
            names = set(n for n, _ in tts_pairs)
            on_disk = set(x.name for x in t_audio)
            print("audio files on disk: %d | referenced by metadata: %d | matched: %d"
                  % (len(on_disk), len(names), len(names & on_disk)))
            print("referenced but missing on disk:", len(names - on_disk))
            print("on disk but not in any mapping:", len(on_disk - names))
        print("")
        for s_ in tts_txt[:10]:
            print("  -", s_[:160])
        TTS_VOCAB = collections.Counter(t.lower() for s_ in tts_txt
                                        for t in re.findall(RE_TOKEN, s_))
        print("\nvocab:", len(TTS_VOCAB), "| top 30:", TTS_VOCAB.most_common(30))
        tchars = collections.Counter("".join(tts_txt))
        print("non-ASCII chars:", [(c, tchars[c]) for c in sorted(tchars) if ord(c) > 127][:30])
    else:
        print("\n!! no transcript column detected - inspect the metadata dumps above manually")

    rec("tts_central",
        n_files=len(tf), by_ext=jsonable(dict(collections.Counter(p.suffix.lower() for p in tf))),
        n_audio=len(t_audio), metadata_files=[_rel(p, TTS_DS) for p in t_meta[:20]],
        clip_duration=tts_dur, n_audio_dirs=len(parents),
        dir_examples=[k for k, _ in parents.most_common(10)],
        n_transcript_rows=len(tts_txt), n_paired=len(tts_pairs),
        tts_vocab_size=len(TTS_VOCAB), top_tts_tokens=TTS_VOCAB.most_common(30),
        unextracted_archives=[Path(p).name for p in t_arch[:10]])

## 8 — Code-switching in Jember

The decisive analysis. Estimates how much of the corpus is Indonesian versus Javanese, at token and utterance level, using the Common Voice Indonesian corpus itself as the Indonesian lexicon. Also lists the vocabulary Indonesian data cannot teach.

In [ ]:
# ============================================================
# CELL 8 - CODE-SWITCHING IN JEMBER
# No external downloads. Two self-contained signals:
#   (a) attestation in the Common Voice Indonesian corpus -> "ID_attested"
#   (b) a high-precision Javanese/Madurese function-word list
# This is an ESTIMATE, not gold annotation. It is directional evidence for
# whether mixing Indonesian data into training is defensible.
# ============================================================
head("CODE-SWITCHING - METHOD")
ID_LEX = set(w for w, n in CV_VOCAB.items() if n >= 2)   # freq>=2 drops CV typos
print("Indonesian lexicon from Common Voice (freq>=2):", len(ID_LEX), "types")

JV_MARKERS = set("""
ora nggak ngono ngene iku kuwi kui aku kowe awakmu sampeyan panjenengan sing seng karo
wis wes durung arep ape ayo dhewe dewe banjur mergo merga soale lha lho
ta rek cak nek yen menawa mung mek wae ae bae sithik akeh apik elek gedhe cilik
piye pripun kepiye opo sopo sapa ngendi kapan pirang siji loro telu papat lima
mangan ngombe turu lungo lunga mulih omah wong bocah cah arek konco kanca
dadi dados nggih inggih boten mboten menika punika kula
mbok ndhuk nduk mas mbak kang
ngapunten matur nuwun suwun sugeng rahayu
malah nemen paran dhika kakeh engko mangke
""".split())
print("Javanese/Madurese marker list:", len(JV_MARKERS), "types")

def classify(tok):
    t = tok.lower().strip("-").strip("'")
    if not t:            return "skip"
    if t in JV_MARKERS:  return "JV"
    if t in ID_LEX:      return "ID_attested"
    return "OOV_id"

head("CODE-SWITCHING - TOKEN-LEVEL")
cnt = collections.Counter(classify(t) for lst in J["tok"] for t in lst)
tot = sum(v for k, v in cnt.items() if k != "skip")
for k in ["JV", "ID_attested", "OOV_id"]:
    print("%-12s %6d  %5.1f %%" % (k, cnt[k], 100 * cnt[k] / max(tot, 1)))
print("")
print("read: JV = confident Javanese function words")
print("      ID_attested = also appears in Indonesian CV (may be shared vocabulary)")
print("      OOV_id = never seen in Indonesian CV")
print("      -> JV + OOV_id is the upper bound on Javanese-specific material.")

head("CODE-SWITCHING - UTTERANCE-LEVEL")
def utt_profile(tokens):
    c = collections.Counter(classify(t) for t in tokens)
    n = sum(v for k, v in c.items() if k != "skip")
    if n == 0: return None
    return (c["JV"] / n, c["OOV_id"] / n, c["ID_attested"] / n, n)

prof = list(J["tok"].map(utt_profile))
P = pd.DataFrame([p for p in prof if p], columns=["jv_r", "oov_r", "id_r", "n"])
P["jv_side"] = P["jv_r"] + P["oov_r"]
print("segments scored:", len(P))
print("")
print("distribution of Javanese-side token ratio per segment:")
print(P["jv_side"].describe(percentiles=[.1, .25, .5, .75, .9]).round(3).to_string())
bins = [0, .01, .2, .4, .6, .8, 1.01]
labs = ["0pct pure-ID", "0-20pct", "20-40pct", "40-60pct", "60-80pct", "80-100pct"]
b = pd.cut(P["jv_side"], bins=bins, labels=labs, right=False).value_counts().sort_index()
print("")
print("segments by Javanese-side ratio:")
for k, v in b.items():
    print("   %-16s %5d  %5.1f %%" % (k, v, 100 * v / len(P)))
mixed = P[(P["jv_side"] > .15) & (P["jv_side"] < .85) & (P["n"] >= 4)]
print("")
print("INTRA-UTTERANCE MIXED (15-85 pct JV, >=4 tokens): %d segments (%.1f %%)"
      % (len(mixed), 100 * len(mixed) / max(len(P), 1)))
print(">>> a high number here means the task really is code-switched ASR,")
print(">>> and a Javanese-only model will lose WER on the Indonesian spans.")

head("CODE-SWITCHING - EXAMPLES")
J2 = J.reset_index(drop=True).copy()
J2["jv_side"] = [(p[0] + p[1]) if p else np.nan for p in prof]
J2["ntok2"]   = [p[3] if p else 0 for p in prof]
for lo, hi, lbl in [(0.0, .05, "LOW JV (Indonesian-looking)"),
                    (.35, .65, "MIXED (code-switched)"),
                    (.90, 1.01, "HIGH JV (Javanese-looking)")]:
    sub = J2[(J2["jv_side"] >= lo) & (J2["jv_side"] < hi) & (J2["ntok2"] >= 5)]
    print("")
    print("--- %s : %d segments ---" % (lbl, len(sub)))
    for _, r in sub.head(8).iterrows():
        print("   [%.2f] %s" % (r["jv_side"], str(r["text"])[:130]))

head("OOV: JEMBER TOKENS ABSENT FROM COMMON VOICE INDONESIAN")
oov = collections.Counter({w: n for w, n in JEM_VOCAB.items() if w not in ID_LEX})
print("OOV types: %d / %d (%.1f %%) | OOV token mass: %.1f %%"
      % (len(oov), len(JEM_VOCAB), 100 * len(oov) / max(len(JEM_VOCAB), 1),
         100 * sum(oov.values()) / max(sum(JEM_VOCAB.values()), 1)))
print("")
print("top 60 OOV tokens - vocabulary that Common Voice Indonesian cannot teach:")
print(oov.most_common(60))

rec("code_switching",
    method="CV-Indonesian attestation + Javanese marker list; estimate not gold",
    id_lexicon_size=len(ID_LEX),
    token_pct={k: round(100 * cnt[k] / max(tot, 1), 1) for k in ["JV", "ID_attested", "OOV_id"]},
    jv_side_ratio_median=round(float(P["jv_side"].median()), 3),
    segments_by_jv_ratio=jsonable({str(k): int(v) for k, v in b.items()}),
    intra_utterance_mixed_pct=round(100 * len(mixed) / max(len(P), 1), 1),
    oov_types_pct=round(100 * len(oov) / max(len(JEM_VOCAB), 1), 1),
    oov_token_mass_pct=round(100 * sum(oov.values()) / max(sum(JEM_VOCAB.values()), 1), 1),
    top_oov=oov.most_common(60))

## 9 — Cross-dataset comparison, split feasibility, tokenizer fertility

Vocabulary coverage between corpora, the read-vs-spontaneous length gap, a session-grouped validation split proposal, and how badly Whisper's BPE tokenizer shreds Javanese.

In [ ]:
# ============================================================
# CELL 9 - CROSS-DATASET COMPARISON + VALIDATION SPLIT FEASIBILITY
# ============================================================
head("VOCABULARY OVERLAP BETWEEN THE THREE CORPORA")
jv_set = set(JEM_VOCAB); cv_set = set(CV_VOCAB); tt_set = set(TTS_VOCAB)
def cover(a, b, cnt_a):
    # share of corpus-A token MASS covered by corpus-B vocabulary
    m = sum(n for w, n in cnt_a.items() if w in b)
    return 100 * m / max(sum(cnt_a.values()), 1)
print("Jember vocab types : %d" % len(jv_set))
print("CV-id  vocab types : %d" % len(cv_set))
print("TTS-cj vocab types : %d" % len(tt_set))
print("")
print("Jember token mass covered by CV-id vocab  : %.1f %%" % cover(jv_set, cv_set, JEM_VOCAB))
if tt_set:
    print("Jember token mass covered by TTS-cj vocab : %.1f %%" % cover(jv_set, tt_set, JEM_VOCAB))
    print("Jember token mass covered by CV+TTS       : %.1f %%" % cover(jv_set, cv_set | tt_set, JEM_VOCAB))
    print("")
    print("type overlap Jember&TTS : %d" % len(jv_set & tt_set))
    print("TTS-only types (Central Javanese vocabulary absent from Jember): %d" % len(tt_set - jv_set))
    ex = [w for w, _ in TTS_VOCAB.most_common(400) if w not in jv_set][:40]
    print("examples:", ex)
print("")
print(">>> If TTS-cj adds little coverage beyond CV-id, its 7-8 hours are")
print(">>> unlikely to pay for the dialect mismatch. Decide on evidence, not vibes.")

head("UTTERANCE-LENGTH DOMAIN GAP")
comp = []
comp.append(dict(corpus="Jember (spontaneous)", median_s=round(float(J["dur_s"].median()), 2),
                 p95_s=round(float(J["dur_s"].quantile(.95)), 2),
                 median_tok=float(J["ntok"].median())))
cvn = CV["validated"]["sentence"].astype(str).map(lambda s: len(re.findall(RE_TOKEN, s)))
comp.append(dict(corpus="CommonVoice id (read)",
                 median_s=round(float((V["ms"] / 1000).median()), 2),
                 p95_s=round(float((V["ms"] / 1000).quantile(.95)), 2),
                 median_tok=float(cvn.median())))
print(pd.DataFrame(comp).to_string(index=False))
print("")
print(">>> Whisper pads every input to 30 s. Short read-speech clips waste compute")
print(">>> and shift the length prior away from the spontaneous target domain.")

head("VALIDATION SPLIT FEASIBILITY FOR JEMBER")
print("The Jember TSV exposes no speaker column, only an audio-file id.")
print("The safest available unit of grouping is therefore the AUDIO FILE (session).")
print("A random SEGMENT-level split would leak the same session/speaker into both")
print("sides and produce a WER that flatters the model. Do not do it.")
print("")
g = J.dropna(subset=["dur_s"]).groupby("fid")["dur_s"].sum().sort_values(ascending=False)
total_h = g.sum() / 3600
print("annotated total: %.2f h across %d files" % (total_h, len(g)))
for frac in [0.10, 0.15, 0.20]:
    target = g.sum() * frac
    acc, chosen = 0.0, []
    for fid, s in g.sample(frac=1.0, random_state=0).items():   # random file order
        if acc >= target: break
        chosen.append(fid); acc += s
    print("held-out %2d%% -> %3d files, %.2f h (train %.2f h, %d files)"
          % (frac * 100, len(chosen), acc / 3600, (g.sum() - acc) / 3600, len(g) - len(chosen)))
print("")
print("file-level annotated minutes: min=%.1f median=%.1f max=%.1f"
      % (g.min() / 60, g.median() / 60, g.max() / 60))

head("WHISPER TOKENIZER FERTILITY ON JAVANESE vs INDONESIAN")
print("Fertility = BPE tokens per word. High fertility on Javanese means the")
print("tokenizer shreds the language, which inflates decode length and hurts WER.")
try:
    from transformers import WhisperTokenizerFast
    tk = WhisperTokenizerFast.from_pretrained("openai/whisper-small", language="indonesian", task="transcribe")
    def fert(words):
        w = [x for x in words if x]
        n = sum(len(tk.encode(" " + x, add_special_tokens=False)) for x in w)
        return n / max(len(w), 1)
    jv_words = [w for w, n in JEM_VOCAB.most_common(3000)]
    cv_words = [w for w, n in CV_VOCAB.most_common(3000)]
    oov_words = [w for w, n in collections.Counter(
        {w: n for w, n in JEM_VOCAB.items() if w not in ID_LEX}).most_common(1000)]
    f_jv, f_cv, f_oov = fert(jv_words), fert(cv_words), fert(oov_words)
    print("")
    print("fertility Jember top-3k words        : %.2f tokens/word" % f_jv)
    print("fertility CommonVoice-id top-3k words: %.2f tokens/word" % f_cv)
    print("fertility Jember OOV top-1k words    : %.2f tokens/word" % f_oov)
    worst = sorted(((len(tk.encode(" " + w, add_special_tokens=False)), w) for w in oov_words),
                   reverse=True)[:25]
    print("")
    print("worst-shredded Javanese words:", worst)
    unk = tk.unk_token_id
    bad = [w for w in jv_words if unk is not None and unk in tk.encode(w, add_special_tokens=False)]
    print("words hitting UNK:", len(bad), bad[:20])
    rec("tokenizer", whisper_fertility_jember=round(f_jv, 2),
        whisper_fertility_cv_id=round(f_cv, 2), whisper_fertility_jember_oov=round(f_oov, 2),
        worst_shredded=[[w, n] for n, w in worst[:20]])
except Exception as e:
    print("")
    print("SKIPPED:", type(e).__name__, e)
    print(">>> turn on Internet in the notebook settings sidebar and re-run this cell.")
    rec("tokenizer", error=str(e)[:200])

rec("cross_dataset",
    jember_mass_covered_by_cv_pct=round(cover(jv_set, cv_set, JEM_VOCAB), 1),
    jember_mass_covered_by_tts_pct=round(cover(jv_set, tt_set, JEM_VOCAB), 1) if tt_set else None,
    jember_mass_covered_by_cv_plus_tts_pct=round(cover(jv_set, cv_set | tt_set, JEM_VOCAB), 1) if tt_set else None,
    length_comparison=jsonable(comp),
    jember_annotated_h=round(float(total_h), 2), jember_n_sessions=int(len(g)),
    session_minutes_median=round(float(g.median() / 60), 1))

## 10 — Digest for handoff

Run last. Copy the marked block back into the chat.

In [ ]:
# ============================================================
# CELL 10 - DIGEST
# Run this last. Copy the ENTIRE block between the BEGIN/END markers
# and paste it back into the Claude conversation. That is the handoff.
# ============================================================
# Completeness gate. A digest that silently lost half its sections is worse than
# no digest at all - it looks authoritative while being wrong.
EXPECTED = ["env", "jember", "jember_audio", "jember_text", "jember_orthography",
            "common_voice", "tts_central", "code_switching", "cross_dataset", "tokenizer"]
missing = [k for k in EXPECTED if k not in FINDINGS]
if missing:
    print("!" * 70)
    print("INCOMPLETE DIGEST - missing sections: %s" % missing)
    print("Cause is almost always a partial re-run: cell 1 resets FINDINGS,")
    print("so any cell not executed after it contributes nothing.")
    print("FIX: Run All, then run this cell again.")
    print("!" * 70)
    print("")
else:
    print("all %d sections present" % len(EXPECTED))
    print("")

FINDINGS["_meta"] = dict(
    notebook="mozilla-javanese dataset EDA v1",
    generated_for="handoff to Claude - do not edit by hand",
    complete=(not missing), missing_sections=missing)

# Compact on purpose: 800+ pretty-printed lines are miserable to copy out of
# a browser. This is one dense blob instead - still valid JSON to read back.
blob = json.dumps(jsonable(FINDINGS), ensure_ascii=False,
                  separators=(",", ":"), sort_keys=True)
print("===== BEGIN CLAUDE-HANDOFF-JSON =====")
print(blob)
print("===== END CLAUDE-HANDOFF-JSON =====")
print("")
print("size: %.1f KB - select the whole block above and paste it into the chat" % (len(blob) / 1024))

# Persist to the mirrored output tree. The pretty-printed copy is the one meant
# to be read by a human; the compact block above is the one meant to be pasted.
try:
    pretty = json.dumps(jsonable(FINDINGS), ensure_ascii=False, indent=1, sort_keys=True)
    out("dataset_findings.json").write_text(pretty, encoding="utf-8")
    print("saved ->", out("dataset_findings.json"))
    sys.stdout.flush()
    print("saved ->", LOG_PATH)
except Exception as e:
    print("could not save:", e)

# Sweep anything Kaggle dropped loose in /kaggle/working into _kaggle-artifacts,
# so the downloaded archive contains only the mirrored tree. Files Kaggle writes
# at save time (after this cell) will still land loose - move those by hand.
try:
    moved = []
    for q in WORKING.iterdir():
        if q.is_file():
            q.replace(ARTIFACT_DIR / q.name); moved.append(q.name)
    if moved:
        print("swept into _kaggle-artifacts:", moved)
except Exception as e:
    print("sweep skipped:", e)

print("")
print("DOWNLOAD: Output tab -> download all. The archive already contains")
print("  findings/%s/ ... so unzip it onto the repo root and it merges." % NOTEBOOK_ID)

head("OPEN QUESTIONS THIS NOTEBOOK COULD NOT ANSWER")
for q in [
    "Speaker identity in Jember: is one mp3 = one speaker, one dialogue, or one session?",
    "  -> without this, a speaker-disjoint validation split cannot be guaranteed.",
    "Dialect label per Jember file: Jember vs Pandhalungan - is it recorded anywhere?",
    "Madurese content: the corpus description lists it, but no marker column exists.",
    "Audio quality: SNR, clipping, overlapped speech. Needs signal analysis, not metadata.",
    "Competition test-set format: segmented utterances or full recordings to be segmented?",
]:
    print(" -", q)